In [1]:
!pip install kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"poorvikah23","key":"0bf72b5d6f4d3ceb28b3653604213825"}'}

In [3]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)
print("Kaggle setup done!")

Kaggle setup done!


In [4]:
!kaggle datasets download -d mohamedmustafa/real-life-violence-situations-dataset

Dataset URL: https://www.kaggle.com/datasets/mohamedmustafa/real-life-violence-situations-dataset
License(s): copyright-authors
100% 3.58G/3.58G [00:35<00:00, 108MB/s]



In [5]:
import zipfile

with zipfile.ZipFile("real-life-violence-situations-dataset.zip", 'r') as zip_ref:
  zip_ref.extractall("dataset")

print("Extraction done!")

Extraction done!


In [6]:
import os

for folder in os.listdir("dataset"):
  print(folder)

real life violence situations
Real Life Violence Dataset


In [7]:
for folder in os.listdir("dataset/Real Life Violence Dataset"):
  print(folder)

Violence
NonViolence


In [8]:
violence_videos = os.listdir("dataset/Real Life Violence Dataset/Violence")
nonviolence_videos = os.listdir("dataset/Real Life Violence Dataset/NonViolence")

print("Violence videos:", len(violence_videos))
print("NonViolence videos:", len(nonviolence_videos))

Violence videos: 1000
NonViolence videos: 1000


In [9]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.9 MB/s eta 0:00:00


In [10]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print("YOLO model loaded!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO model loaded!


In [11]:
# Get first video from Violence folder
video_path = "dataset/Real Life Violence Dataset/Violence/" + violence_videos[0]

print("Testing on video:", violence_videos[0])

Testing on video: V_285.mp4


In [12]:
results = model.predict(
    source=video_path,
    save=True,
    conf=0.5
)

print("Detection done!")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/156) /content/dataset/Real Life Violence Dataset/Violence/V_285.mp4: 384x640 5 persons, 934.7ms
video 1/1 (frame 2/156) /content/dataset/Real Life Violence Dataset/Violence/V_285.mp4: 384x640 5 persons, 394.7ms
video 1/1 (frame 3/156) /content/dataset/Real Life Violence Dataset/Violence/V_285.mp4: 384x640 5 persons, 149.4ms
video 1/1 (frame 4/156) /content/dataset/Real Life Violence Dataset/Violence/V_285.mp4: 384x640 5 persons, 152.5m

In [13]:
import glob

# Search for any video file saved
files = glob.glob("runs/detect/predict/*")
for f in files:
    print(f)

runs/detect/predict/V_285.avi


In [14]:
import subprocess

subprocess.run([
    "ffmpeg", "-i",
    "runs/detect/predict/V_285.avi",
    "runs/detect/predict/V_285_output.mp4",
    "-y"
])

print("Converted to MP4!")

Converted to MP4!


In [15]:
from IPython.display import HTML
from base64 import b64encode

with open("runs/detect/predict/V_285_output.mp4", "rb") as f:
    video_data = f.read()

video_base64 = b64encode(video_data).decode()

HTML(f"""
<video width="640" height="384" controls>
<source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
</video>
""")

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
!pip install tensorflow opencv-python-headless

In [18]:
import cv2
import numpy as np
import os
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

In [19]:
def extract_frames(video_path, max_frames=10):
    frames = []
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total // max_frames, 1)

    count = 0
    frame_no = 0
    while count < max_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (224, 224))
        frame = frame / 255.0
        frames.append(frame)
        frame_no += step
        count += 1

    cap.release()
    return frames

print("Function created!")

Function created!


In [20]:
X = []  # frames
y = []  # labels

violence_path = "dataset/Real Life Violence Dataset/Violence/"
nonviolence_path = "dataset/Real Life Violence Dataset/NonViolence/"

print("Processing Violence videos...")
for video in os.listdir(violence_path)[:100]:  # 100 videos for speed
    frames = extract_frames(violence_path + video)
    for frame in frames:
        X.append(frame)
        y.append(1)  # 1 = Violence

print("Processing NonViolence videos...")
for video in os.listdir(nonviolence_path)[:100]:  # 100 videos for speed
    frames = extract_frames(nonviolence_path + video)
    for frame in frames:
        X.append(frame)
        y.append(0)  # 0 = NonViolence

X = np.array(X)
y = np.array(y)

print("Total frames:", len(X))
print("Done!")

Processing Violence videos...
Processing NonViolence videos...
Total frames: 2000
Done!


In [21]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Violence frames:", sum(y == 1))
print("NonViolence frames:", sum(y == 0))

X shape: (2000, 224, 224, 3)
y shape: (2000,)
Violence frames: 1000
NonViolence frames: 1000


In [22]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model_violence = Model(inputs=base_model.input, outputs=output)

model_violence.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model built!")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model built!


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

history = model_violence.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

print("Training done!")

Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step - accuracy: 0.8600 - loss: 0.3113 - val_accuracy: 0.9225 - val_loss: 0.2016
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 93s 2s/step - accuracy: 0.9613 - loss: 0.1059 - val_accuracy: 0.9475 - val_loss: 0.1495
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.9850 - loss: 0.0524 - val_accuracy: 0.9725 - val_loss: 0.0834
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.9925 - loss: 0.0308 - val_accuracy: 0.9750 - val_loss: 0.0684
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step - accuracy: 0.9981 - loss: 0.0188 - val_accuracy: 0.9750 - val_loss: 0.0613
Training done!


In [24]:
print(X.shape)
print("Model ready:", model_violence)

(2000, 224, 224, 3)
Model ready: <Functional name=functional, built=True>


In [25]:
loss, accuracy = model_violence.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test Loss: {loss:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.9750 - loss: 0.0613
Test Accuracy: 97.50%
Test Loss: 0.0613


In [26]:
model_violence.save("violence_model.h5")
print("Model saved!")

Model saved!


In [27]:
import shutil

shutil.copy("violence_model.h5", "/content/drive/MyDrive/violence_model.h5")
print("Model saved to Google Drive!")

Model saved to Google Drive!


In [28]:
shutil.copy("/root/.kaggle/kaggle.json",
"/content/drive/MyDrive/kaggle.json")
print("kaggle.json backed up!")

kaggle.json backed up!


In [29]:
def detect_violence(video_path):
    cap = cv2.VideoCapture(video_path)
    violence_detected = False
    snapshot_path = None

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        if frame_count % 10 != 0:  # Check every 10th frame
            continue

        # YOLO detects persons
        results = model.predict(frame, conf=0.5, verbose=False)
        persons = [r for r in results[0].boxes.cls if r == 0]

        if len(persons) >= 2:  # At least 2 people
            # MobileNet checks violence
            resized = cv2.resize(frame, (224, 224)) / 255.0
            input_frame = np.expand_dims(resized, axis=0)
            prediction = model_violence.predict(input_frame, verbose=False)[0][0]

            if prediction > 0.7:  # 70% confident = violence
                violence_detected = True
                snapshot_path = f"snapshot_{frame_count}.jpg"
                cv2.imwrite(snapshot_path, frame)
                print(f"VIOLENCE DETECTED! Confidence: {prediction:.2f}")
                break

    cap.release()
    return violence_detected, snapshot_path

print("Function ready!")

Function ready!


In [30]:
test_video = "dataset/Real Life Violence Dataset/Violence/" + violence_videos[0]

print("Testing on:", violence_videos[0])
detected, snapshot = detect_violence(test_video)

if detected:
    print("✅ VIOLENCE DETECTED!")
    print("Snapshot saved:", snapshot)
else:
    print("❌ No violence detected")

Testing on: V_285.mp4
VIOLENCE DETECTED! Confidence: 0.98
✅ VIOLENCE DETECTED!
Snapshot saved: snapshot_10.jpg


In [31]:
import requests
import json
import base64

def send_alert(snapshot_path):
    # Read snapshot and convert to base64
    with open(snapshot_path, "rb") as f:
        snapshot_base64 = base64.b64encode(f.read()).decode()

    alert_data = {
        "threat_detected": True,
        "confidence": 0.98,
        "snapshot": snapshot_base64,
        "camera": "Camera 1",
        "time": "Now"
    }

    # Save alert to JSON file for Teammate B
    with open("alert.json", "w") as f:
        json.dump(alert_data, f)

    print("Alert saved! Ready to send to Teammate B!")

send_alert("snapshot_10.jpg")

Alert saved! Ready to send to Teammate B!


In [32]:
import shutil

shutil.copy("alert.json", "/content/drive/MyDrive/alert.json")
shutil.copy("snapshot_10.jpg", "/content/drive/MyDrive/snapshot_10.jpg")
print("Everything saved to Drive!")

Everything saved to Drive!
